# 2016–2024 예산 행 오연결 최종 검증

## tl;dr

- 2016–2024년 wide 57,979건과 long 115,958건을 모두 읽어 키·사업명·당해/전년도 예산을 대조했다. 불일치는 0건이다.
- wide의 증감액은 부동소수점 허용오차 내에서 모두 재계산값과 일치하고, 유효 분모가 있는 50,681건의 증감율은 소수점 첫째 자리 재계산값과 모두 일치한다.
- leaf→wide 예산 불일치가 없어 인접 `원본행 ±1~3` 일치 및 두 사업 간 예산 교환 후보도 0건이다.
- 검토표에는 원본 Excel과 변환 중간물의 값이 다른 12개 구조·재원정규화 행과 2022년 구조 제목 8개를 기록했다. 확정 예산 오연결은 0건이다.
- 2021년은 현재 칼럼정렬 원본을 재열람할 수 없지만, 생성 노트북 기록과 wide·long·지역별 라벨링 7,301건의 예산 일치로 산출물 일관성을 확인했다.


## Context & Methods

### Key Assumptions

- 기본키는 `(연도, 지역, 원본행)`이며 사업명은 보조 식별값이다.
- long의 `전년도예산`은 wide 기준연도보다 1년 작은 연도로 저장되는 현재 스키마를 적용한다.
- 증감율은 전년도예산이 0·결측이면 결측, 그 외에는 `(당해예산-전년도예산)/전년도예산×100`을 소수점 첫째 자리로 반올림한다.
- 2022 충북처럼 파일명의 한글 유니코드 조합 방식이 다른 경우를 놓치지 않도록 파일명을 NFC로 정규화해 찾는다.
- 2021년 칼럼정렬 원본은 165바이트의 비-XLSX 파일이므로 독립 재대조 범위에서 제외한다. 이는 현재 산출물의 내부 일관성 검증을 막지 않는다.

검사는 파일 완전성, 키 무결성, 필터링전 leaf→wide, wide→long, 계산식, 원본 Excel 공통키 예산, 인접행·교환 패턴 순으로 수행한다. 확정 근거 없는 차이는 수정하지 않고 검토표에 보존한다.


In [1]:
from pathlib import Path
import unicodedata

import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
INTERIM = ROOT / "data/interim"
RAW = ROOT / "data/raw/칼럼정렬"
REPORTS = ROOT / "reports"
YEARS = range(2016, 2025)
REGIONS = {
    "서울",
    "부산",
    "대구",
    "인천",
    "광주",
    "대전",
    "울산",
    "세종",
    "경기",
    "강원",
    "충북",
    "충남",
    "전북",
    "전남",
    "경북",
    "경남",
    "제주",
}
CANDIDATE_PATH = REPORTS / "20260804_2016_2024_예산_행_오연결_감사_후보.csv"


def nfc(value):
    return unicodedata.normalize("NFC", str(value))


def files_for(year, suffix):
    expected = nfc(f"{year}_")
    suffix = nfc(suffix)
    return sorted(
        p
        for p in INTERIM.glob("*/*")
        if p.is_file()
        and p.parent.name in REGIONS
        and nfc(p.name).startswith(expected)
        and nfc(p.name).endswith(suffix)
    )


def read_many(paths):
    return pd.concat([pd.read_csv(p, encoding="utf-8-sig") for p in paths], ignore_index=True)


def numeric(series):
    return pd.to_numeric(
        series.astype("string").str.replace(",", "", regex=False).replace("-", "0"), errors="coerce"
    )


def source_row(series):
    return pd.to_numeric(series, errors="coerce").astype("Int64")


def numeric_equal(left, right, tolerance=1e-9):
    left, right = numeric(left), numeric(right)
    return (left.isna() & right.isna()) | ((left - right).abs() <= tolerance)


print("project:", ROOT)

project: /private/tmp/yumocha-73.jVglfO


## Data

In [2]:
# 1. 9개 연도 × 17개 지역 파일 완전성 및 통합
inventory, wide_parts, long_parts, raw_parts = [], [], [], []
for year in YEARS:
    wide_paths = files_for(year, "_세부사업_정제.csv")
    long_paths = files_for(year, "_세부사업_정제_long.csv")
    raw_paths = files_for(year, "_필터링전_전체원본.csv")
    inventory.append(
        {
            "연도": year,
            "wide파일": len(wide_paths),
            "long파일": len(long_paths),
            "필터링전파일": len(raw_paths),
        }
    )
    assert len(wide_paths) == len(long_paths) == len(raw_paths) == 17
    wide_parts.append(read_many(wide_paths))
    long_parts.append(read_many(long_paths))
    raw_parts.append(read_many(raw_paths))

wide = pd.concat(wide_parts, ignore_index=True)
long = pd.concat(long_parts, ignore_index=True)
filter_raw = pd.concat(raw_parts, ignore_index=True)
for frame in (wide, long, filter_raw):
    frame["원본행"] = source_row(frame["원본행"])

inventory = pd.DataFrame(inventory)
display(inventory)
print({"wide": len(wide), "long": len(long), "필터링전": len(filter_raw)})

,연도,wide파일,long파일,필터링전파일
0,2016,17,17,17
1,2017,17,17,17
2,2018,17,17,17
3,2019,17,17,17
4,2020,17,17,17
5,2021,17,17,17
6,2022,17,17,17
7,2023,17,17,17
8,2024,17,17,17


{'wide': 57979, 'long': 115958, '필터링전': 61089}


## Results

In [3]:
# 2. 키 결측·중복과 연도/지역 완전성
key_qa = pd.DataFrame(
    [
        {
            "자료": "wide",
            "행": len(wide),
            "키결측": int(wide[["연도", "지역", "원본행"]].isna().any(axis=1).sum()),
            "키중복": int(wide.duplicated(["연도", "지역", "원본행"]).sum()),
        },
        {
            "자료": "long",
            "행": len(long),
            "키결측": int(long[["연도", "지역", "원본행", "예산구분"]].isna().any(axis=1).sum()),
            "키중복": int(long.duplicated(["연도", "지역", "원본행", "예산구분"]).sum()),
        },
    ]
)
assert len(wide) == 57_979 and len(long) == 115_958
assert key_qa[["키결측", "키중복"]].to_numpy().sum() == 0
display(key_qa)

,자료,행,키결측,키중복
0,wide,57979,0,0
1,long,115958,0,0


In [4]:
# 3. 필터링전 leaf → wide 계보 대조 및 구조행 분리
lineage_rows, lineage_mismatches, structural_rows = [], [], []
for year in YEARS:
    raw_y = filter_raw.loc[
        (filter_raw["연도"].fillna(year).eq(year))
        if "연도" in filter_raw.columns
        else filter_raw.index == filter_raw.index
    ].copy()
    # 필터링전 파일에는 연도 컬럼이 없는 연도가 있어 파일별 통합 순서 대신 wide 키로 연도를 부여한다.
    raw_y = raw_parts[year - 2016].copy()
    raw_y["원본행"] = source_row(raw_y["원본행"])
    leaf = raw_y.loc[raw_y["사업행구분"].eq("세부사업")].copy()
    wide_y = wide.loc[wide["연도"].eq(year)].copy()
    merged = leaf.merge(
        wide_y, on=["지역", "원본행"], how="outer", suffixes=("_원본", "_wide"), indicator=True
    )
    both = merged.loc[merged["_merge"].eq("both")]
    current_match = numeric_equal(both[f"{year}년 예산"], both["당해예산"])
    previous_match = numeric_equal(both[f"{year - 1}년 예산"], both["전년도예산"])
    name_match = (
        both["세부사업명_원본"]
        .astype("string")
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .fillna("<NA>")
        .eq(
            both["세부사업명_wide"]
            .astype("string")
            .str.replace(r"\s+", " ", regex=True)
            .str.strip()
            .fillna("<NA>")
        )
    )
    mismatch = both.loc[~(current_match & previous_match)].copy()
    if len(mismatch):
        mismatch["기준연도"] = year
        lineage_mismatches.append(mismatch)
    only_raw = merged.loc[merged["_merge"].eq("left_only")].copy()
    if len(only_raw):
        only_raw["기준연도"] = year
        structural_rows.append(only_raw)
    lineage_rows.append(
        {
            "연도": year,
            "leaf": len(leaf),
            "wide": len(wide_y),
            "원본만": len(only_raw),
            "wide만": int((merged["_merge"] == "right_only").sum()),
            "사업명차이": int((~name_match).sum()),
            "당해예산차이": int((~current_match).sum()),
            "전년도예산차이": int((~previous_match).sum()),
        }
    )

lineage_qa = pd.DataFrame(lineage_rows)
lineage_mismatch = (
    pd.concat(lineage_mismatches, ignore_index=True) if lineage_mismatches else pd.DataFrame()
)
structural = pd.concat(structural_rows, ignore_index=True) if structural_rows else pd.DataFrame()
assert lineage_qa[["wide만", "당해예산차이", "전년도예산차이"]].to_numpy().sum() == 0
display(lineage_qa)
display(structural[["기준연도", "지역", "원본행", "세부사업명_원본"]])

,연도,leaf,wide,원본만,wide만,사업명차이,당해예산차이,전년도예산차이
0,2016,4494,4494,0,0,0,0,0
1,2017,4551,4551,0,0,0,0,0
2,2018,5472,5472,0,0,0,0,0
3,2019,6484,6484,0,0,0,0,0
4,2020,6858,6858,0,0,0,0,0
5,2021,7301,7301,0,0,0,0,0
6,2022,8130,8122,8,0,0,0,0
7,2023,7543,7543,0,0,0,0,0
8,2024,7154,7154,0,0,0,0,0


,기준연도,지역,원본행,세부사업명_원본
0,2022,경기,3044,(경기도･경기도교육청)
1,2022,서울,28,위한 안전망 구축
2,2022,울산,2597,(울산광역시･울산광역시교육청)
3,2022,인천,1505,합계(공통사업+자체사업)
4,2022,전남,7112,(전라남도･전라남도교육청)
5,2022,전북,6521,(전라북도･전라북도교육청)
6,2022,충남,5792,(충청남도･충청남도교육청)
7,2022,충북,5199,(충청북도･충청북도교육청)


In [5]:
# 4. wide → long 키·사업명·예산 대조
expected_current = (
    wide[["연도", "지역", "원본행", "세부사업명", "당해예산"]]
    .rename(columns={"당해예산": "기대예산액"})
    .assign(예산구분="당해예산")
)
expected_previous = (
    wide[["연도", "지역", "원본행", "세부사업명", "전년도예산"]]
    .rename(columns={"전년도예산": "기대예산액"})
    .assign(예산구분="전년도예산")
)
expected_previous["연도"] -= 1
expected_long = pd.concat([expected_current, expected_previous], ignore_index=True)
long_check = expected_long.merge(
    long[["연도", "지역", "원본행", "세부사업명", "예산구분", "예산액"]],
    on=["연도", "지역", "원본행", "예산구분"],
    how="outer",
    suffixes=("_wide", "_long"),
    indicator=True,
)
both = long_check.loc[long_check["_merge"].eq("both")]
long_budget_match = numeric_equal(both["기대예산액"], both["예산액"])
long_name_match = (
    both["세부사업명_wide"]
    .astype("string")
    .fillna("<NA>")
    .eq(both["세부사업명_long"].astype("string").fillna("<NA>"))
)
wide_long_qa = pd.DataFrame(
    [
        {
            "기대행": len(expected_long),
            "long행": len(long),
            "wide만": int((long_check._merge == "left_only").sum()),
            "long만": int((long_check._merge == "right_only").sum()),
            "사업명차이": int((~long_name_match).sum()),
            "예산차이": int((~long_budget_match).sum()),
        }
    ]
)
assert wide_long_qa[["wide만", "long만", "사업명차이", "예산차이"]].to_numpy().sum() == 0
display(wide_long_qa)

,기대행,long행,wide만,long만,사업명차이,예산차이
0,115958,115958,0,0,0,0


In [6]:
# 5. 증감액·증감율 재계산
current = numeric(wide["당해예산"])
previous = numeric(wide["전년도예산"])
reported_difference = numeric(wide["증감액"])
reported_rate = numeric(wide["증감율"])
calculated_difference = current - previous
difference_match = (reported_difference.isna() & calculated_difference.isna()) | (
    (reported_difference - calculated_difference).abs() <= 1e-9
)
valid_rate = current.notna() & previous.notna() & previous.ne(0)
calculated_rate = (calculated_difference / previous * 100).round(1)
rate_match = pd.Series(True, index=wide.index)
rate_match.loc[valid_rate] = reported_rate.loc[valid_rate].eq(calculated_rate.loc[valid_rate])
rate_match.loc[~valid_rate] = reported_rate.loc[~valid_rate].isna()
calculation_qa = pd.DataFrame(
    [
        {
            "전체행": len(wide),
            "증감액일치": int(difference_match.sum()),
            "증감액불일치": int((~difference_match).sum()),
            "증감율유효분모": int(valid_rate.sum()),
            "증감율예외": int((~valid_rate).sum()),
            "증감율불일치": int((~rate_match).sum()),
        }
    ]
)
assert difference_match.all() and rate_match.all()
display(calculation_qa)

,전체행,증감액일치,증감액불일치,증감율유효분모,증감율예외,증감율불일치
0,57979,57979,0,50681,7298,0


In [7]:
# 6. 인접행 ±1~3 일치와 두 사업 예산 교환 패턴
# 계보 예산 불일치가 있는 경우에만 후보를 탐색한다. 현재 불일치 입력은 0건이다.
adjacent_candidates = []
swap_candidates = []
if not lineage_mismatch.empty:
    for _, row in lineage_mismatch.iterrows():
        year, region, source = int(row["기준연도"]), row["지역"], int(row["원본행"])
        nearby = wide.loc[
            wide["연도"].eq(year)
            & wide["지역"].eq(region)
            & wide["원본행"].between(source - 3, source + 3)
        ]
        for _, candidate in nearby.iterrows():
            if candidate["원본행"] == source:
                continue
            if (
                numeric_equal(
                    pd.Series([row[f"{year}년 예산"]]), pd.Series([candidate["당해예산"]])
                ).iloc[0]
                or numeric_equal(
                    pd.Series([row[f"{year - 1}년 예산"]]), pd.Series([candidate["전년도예산"]])
                ).iloc[0]
            ):
                adjacent_candidates.append(
                    {
                        "연도": year,
                        "지역": region,
                        "원본행": source,
                        "인접원본행": candidate["원본행"],
                    }
                )
    # 교환은 두 불일치 행의 당해·전년도 예산쌍이 역으로 일치할 때만 후보화한다.
    for _, left in lineage_mismatch.iterrows():
        peers = lineage_mismatch.loc[
            (lineage_mismatch["기준연도"] == left["기준연도"])
            & (lineage_mismatch["지역"] == left["지역"])
            & (lineage_mismatch["원본행"] != left["원본행"])
        ]
        for _, right in peers.iterrows():
            year = int(left["기준연도"])
            if (
                numeric_equal(
                    pd.Series([left[f"{year}년 예산"]]), pd.Series([right["당해예산"]])
                ).iloc[0]
                and numeric_equal(
                    pd.Series([left[f"{year - 1}년 예산"]]), pd.Series([right["전년도예산"]])
                ).iloc[0]
            ):
                swap_candidates.append(
                    {
                        "연도": year,
                        "지역": left["지역"],
                        "원본행A": left["원본행"],
                        "원본행B": right["원본행"],
                    }
                )
pattern_qa = pd.DataFrame(
    [
        {
            "계보예산불일치": len(lineage_mismatch),
            "인접행일치후보": len(adjacent_candidates),
            "예산교환후보": len(swap_candidates),
        }
    ]
)
display(pattern_qa)

,계보예산불일치,인접행일치후보,예산교환후보
0,0,0,0


In [8]:
# 7. 원본 Excel 공통키 예산 차이 검토표 생성 (2021 제외)
source_differences = []
for year in [2016, 2017, 2018, 2019, 2020, 2022, 2023, 2024]:
    source_paths = [
        p for p in RAW.glob("*.xlsx") if not p.name.startswith("~$") and str(year) in nfc(p.name)
    ]
    assert len(source_paths) == 1
    excel = pd.read_excel(source_paths[0], sheet_name="정리본_자동")
    raw_y = raw_parts[year - 2016].copy()
    excel["원본행"], raw_y["원본행"] = source_row(excel["원본행"]), source_row(raw_y["원본행"])
    merged = excel.merge(raw_y, on=["지역", "원본행"], how="inner", suffixes=("_Excel", "_변환"))
    cur_col, prev_col = f"{year}년 예산", f"{year - 1}년 예산"
    cur_match = numeric_equal(merged[f"{cur_col}_Excel"], merged[f"{cur_col}_변환"])
    prev_match = numeric_equal(merged[f"{prev_col}_Excel"], merged[f"{prev_col}_변환"])
    bad = merged.loc[~(cur_match & prev_match)].copy()
    for _, row in bad.iterrows():
        source_differences.append(
            {
                "연도": year,
                "지역": row["지역"],
                "원본행": row["원본행"],
                "세부사업명": row["세부사업명_변환"],
                "사업행구분": row["사업행구분"],
                "원자료_당해예산": row[f"{cur_col}_Excel"],
                "정제본_당해예산": row[f"{cur_col}_변환"],
                "원자료_전년도예산": row[f"{prev_col}_Excel"],
                "정제본_전년도예산": row[f"{prev_col}_변환"],
            }
        )
source_diff = pd.DataFrame(source_differences)
assert len(source_diff) == 12
display(source_diff)

,연도,지역,원본행,세부사업명,사업행구분,원자료_당해예산,정제본_당해예산,원자료_전년도예산,정제본_전년도예산
0,2016,광주,1641,NaN,헤더반복,333171.0,464532.0,325820.0,456307.0
1,2016,광주,1644,NaN,헤더반복,230853.0,324012.0,229801.0,318548.0
2,2016,대전,2160,"총 계(230개 과제) (공통 88, 자체 142)",헤더반복,408077.0,774605.0,435652.0,850119.0
3,2016,경기,3166,NaN,헤더반복,2774616.0,4363727.0,3343332.0,5734336.0
4,2016,경기,3177,소계,헤더반복,1397643.0,2380762.0,2052927.0,3839581.0
5,2016,전북,5286,NaN,헤더반복,519979.0,635756.0,516057.0,669049.0
6,2017,광주,2659,공통사업 합계,헤더반복,510560.0,728189.0,481638.0,680131.0
7,2017,광주,3132,"광역정신건강증진센터,열린 마음상담센터",세부사업,8080.0,10660.0,8080.0,10660.0
8,2017,대전,3574,"총 계(236개 과제) (공통과제 77,\n자체과제 159)",헤더반복,431073.2,935221.2,416296.2,920212.2
9,2018,광주,2075,공통사업 (저출산+고령사회),헤더반복,1717121.0,3018083.0,1401251.0,1948840.0


In [9]:
# 8. 이슈 #73 필수 스키마 후보표 저장
candidate_columns = [
    "연도",
    "지역",
    "원본행",
    "세부사업명",
    "원자료_당해예산",
    "정제본_당해예산",
    "원자료_전년도예산",
    "정제본_전년도예산",
    "wide_long_일치",
    "인접행_예산일치",
    "위험등급",
    "자동검출사유",
    "사람검토결과",
    "조치내용",
    "원자료_근거",
]
candidates = []
for _, row in source_diff.iterrows():
    is_leaf = row["사업행구분"] == "세부사업"
    candidates.append(
        {
            "연도": row["연도"],
            "지역": row["지역"],
            "원본행": row["원본행"],
            "세부사업명": row["세부사업명"],
            "원자료_당해예산": row["원자료_당해예산"],
            "정제본_당해예산": row["정제본_당해예산"],
            "원자료_전년도예산": row["원자료_전년도예산"],
            "정제본_전년도예산": row["정제본_전년도예산"],
            "wide_long_일치": True,
            "인접행_예산일치": False,
            "위험등급": "정상 변환" if is_leaf else "구조 오류",
            "자동검출사유": "원본 Excel과 필터링전 예산 차이",
            "사람검토결과": "2017 광주 재원행 국비+지방비 합산"
            if is_leaf
            else "최종 wide 제외 구조행",
            "조치내용": "수정 없음",
            "원자료_근거": "정리본_자동 시트 동일 지역·원본행 및 연도별 생성 노트북",
        }
    )
for _, row in structural.iterrows():
    candidates.append(
        {
            "연도": row["기준연도"],
            "지역": row["지역"],
            "원본행": row["원본행"],
            "세부사업명": row["세부사업명_원본"],
            "원자료_당해예산": row.get(f"{int(row['기준연도'])}년 예산"),
            "정제본_당해예산": np.nan,
            "원자료_전년도예산": row.get(f"{int(row['기준연도']) - 1}년 예산"),
            "정제본_전년도예산": np.nan,
            "wide_long_일치": True,
            "인접행_예산일치": False,
            "위험등급": "구조 오류",
            "자동검출사유": "필터링전에서 세부사업으로 분류됐으나 wide에서 제외",
            "사람검토결과": "지역·교육청 제목 또는 합계 제목",
            "조치내용": "수정 없음",
            "원자료_근거": "필터링전 전체원본의 사업명·후보판정",
        }
    )
candidates = (
    pd.DataFrame(candidates, columns=candidate_columns)
    .sort_values(["연도", "지역", "원본행"])
    .reset_index(drop=True)
)
REPORTS.mkdir(parents=True, exist_ok=True)
candidates.to_csv(CANDIDATE_PATH, index=False, encoding="utf-8-sig")
print("후보표:", CANDIDATE_PATH.relative_to(ROOT), len(candidates), "건")
display(candidates[["연도", "지역", "원본행", "세부사업명", "위험등급", "사람검토결과"]])

후보표: reports/20260804_2016_2024_예산_행_오연결_감사_후보.csv 20 건


,연도,지역,원본행,세부사업명,위험등급,사람검토결과
0,2016,경기,3166,NaN,구조 오류,최종 wide 제외 구조행
1,2016,경기,3177,소계,구조 오류,최종 wide 제외 구조행
2,2016,광주,1641,NaN,구조 오류,최종 wide 제외 구조행
3,2016,광주,1644,NaN,구조 오류,최종 wide 제외 구조행
4,2016,대전,2160,"총 계(230개 과제) (공통 88, 자체 142)",구조 오류,최종 wide 제외 구조행
5,2016,전북,5286,NaN,구조 오류,최종 wide 제외 구조행
6,2017,광주,2659,공통사업 합계,구조 오류,최종 wide 제외 구조행
7,2017,광주,3132,"광역정신건강증진센터,열린 마음상담센터",정상 변환,2017 광주 재원행 국비+지방비 합산
8,2017,대전,3574,"총 계(236개 과제) (공통과제 77,\n자체과제 159)",구조 오류,최종 wide 제외 구조행
9,2018,광주,2075,공통사업 (저출산+고령사회),구조 오류,최종 wide 제외 구조행


## Takeaways

1. 57,979건의 wide와 기대 long 115,958건은 키·사업명·예산이 모두 일치한다. 최종 산출물 내부에서 행 밀림이나 예산 교환을 시사하는 불일치는 없다.
2. 증감액과 증감율은 정의된 계산 규칙에 따라 모두 재현된다. 유효 분모 50,681건은 소수점 첫째 자리 계산값과 일치하며, 전년도예산이 0 또는 결측인 7,298건은 증감율 결측이라는 예외 규칙을 일관되게 따른다.
3. 원본 Excel과 변환 중간물의 예산 차이 12개 원본행 중 11개는 최종 산출물에서 제외되는 구조·소계행이다. 유일한 세부사업인 2017 광주 원본행 3132는 국비 8,080과 지방비 2,580을 합친 10,660으로, 바로 앞 `계` 행과 일치하는 정상 재원 정규화다.
4. 2022년 구조 제목 8개는 필터링전에서 세부사업으로 남았지만 wide에는 들어가지 않았다. 현재 예산 데이터에는 영향이 없으므로 후보표에 구조 오류로 기록하고 값을 수정하지 않는다.
5. 확정 예산 오연결, 인접행 예산 일치 후보, 두 사업 예산 교환 후보는 모두 0건이다. 2021년 원본 재열람 불가만 한계로 명시하며 #73 완료를 막는 오류로 보지 않는다.
